# LeetCode #72 — Edit Distance: DP hai chuỗi kinh điển

**Mức độ:** Medium · **Chủ đề:** String, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
```

Đây là **khoảng cách Levenshtein** — dùng trong kiểm tra chính tả, so sánh DNA, `diff`... Bài này là "mẫu gốc" của họ **DP trên hai chuỗi**: bảng `dp[i][j]` cho tiền tố `word1[:i]` và `word2[:j]`.

Mục tiêu của notebook:

1. Định nghĩa trạng thái trên **tiền tố** của hai chuỗi.
2. Ba thao tác (chèn, xóa, thay) ↔ ba ô hàng xóm trong bảng.
3. Hàng 0 / cột 0 (chuỗi rỗng) và tối ưu còn hai hàng.
4. Truy vết để in ra **dãy thao tác** cụ thể.

# 1. Bài toán là gì?

Cho hai chuỗi `word1`, `word2`. Trả về **số thao tác ít nhất** để biến `word1` thành `word2`, với ba loại thao tác trên một ký tự:

- **Insert** (chèn) một ký tự.
- **Delete** (xóa) một ký tự.
- **Replace** (thay) một ký tự bằng ký tự khác.

**Constraints (theo đề):**

- `0 <= word1.length, word2.length <= 500`
- `word1`, `word2` gồm chữ cái tiếng Anh viết thường.

---

## Ví dụ

```text
word1 = "horse", word2 = "ros"  ->  3
  horse -> rorse  (thay 'h' bằng 'r')
  rorse -> rose   (xóa 'r')
  rose  -> ros    (xóa 'e')

word1 = "intention", word2 = "execution"  ->  5
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)
2. DP hai chiều (bảng `dp[i][j]`)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 DP hai chiều (bảng `dp[i][j]`)

### Khi nào cần bảng hai chiều?

Khi trạng thái cần **hai** thông số, thường gặp ở:

- **Lưới**: `dp[i][j]` = đáp án cho ô `(i, j)` (số đường đi, tổng nhỏ nhất…).
- **Hai chuỗi**: `dp[i][j]` = đáp án cho **tiền tố** `s[:i]` và `t[:j]` (edit distance, LCS, xen kẽ…).
- **Đoạn con**: `dp[i][j]` = đáp án cho đoạn `s[i..j]` (palindrome…).

### Ví dụ 1: số đường đi trên lưới

Robot ở góc trên-trái, mỗi bước đi **phải** hoặc **xuống**. Có bao nhiêu đường tới ô `(i, j)`? Bước cuối đến từ ô **trên** hoặc ô **trái**:

```text
dp[i][j] = dp[i-1][j] + dp[i][j-1]
hàng 0 và cột 0 đều = 1 (chỉ một cách: đi thẳng)
```

In [3]:
def so_duong(m, n):
    dp = [[0] * n for _ in range(m)]      # KHÔNG dùng [[0] * n] * m
    for i in range(m):
        for j in range(n):
            if i == 0 or j == 0:
                dp[i][j] = 1
            else:
                dp[i][j] = dp[i - 1][j] + dp[i][j - 1]
    return dp

for row in so_duong(3, 5):
    print(" ".join(f"{v:>3}" for v in row))

  1   1   1   1   1
  1   2   3   4   5
  1   3   6  10  15


Thứ tự điền: theo hàng từ trên xuống, trong hàng từ trái sang → khi tính `dp[i][j]`, ô trên và ô trái **đã có**.

**Tiết kiệm bộ nhớ:** hàng `i` chỉ cần hàng `i−1` → giữ một mảng một chiều `row`, cập nhật `row[j] += row[j-1]` (giá trị cũ của `row[j]` chính là ô trên).

In [4]:
def so_duong_mot_hang(m, n):
    row = [1] * n
    for _ in range(1, m):
        for j in range(1, n):
            row[j] += row[j - 1]          # row[j] (cũ) = trên, row[j-1] (mới) = trái
    return row[-1]
print("so_duong_mot_hang(3, 7) =", so_duong_mot_hang(3, 7), "| so_duong(3, 7) =", so_duong(3, 7)[-1][-1])

so_duong_mot_hang(3, 7) = 28 | so_duong(3, 7) = 28


### Ví dụ 2: hai chuỗi — dãy con chung dài nhất (LCS)

`dp[i][j]` = độ dài dãy con chung dài nhất của `s[:i]` và `t[:j]`. Xét ký tự **cuối** của hai tiền tố:

- `s[i-1] == t[j-1]` → dùng nó: `dp[i][j] = dp[i-1][j-1] + 1`.
- khác → bỏ ký tự cuối của một trong hai: `dp[i][j] = max(dp[i-1][j], dp[i][j-1])`.

Hàng 0 và cột 0 ứng với **chuỗi rỗng** → bằng 0. Đó là lý do bảng có kích thước `(n+1) × (m+1)`.

In [5]:
def lcs(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp

s_demo, t_demo = "abcde", "ace"
bang = lcs(s_demo, t_demo)
print("      ''  " + "   ".join(t_demo))
for i, row in enumerate(bang):
    print(f"{('' if i == 0 else s_demo[i - 1])!r:>4} " + " ".join(f"{v:>3}" for v in row))
print("LCS =", bang[-1][-1])

      ''  a   c   e
  ''   0   0   0   0
 'a'   0   1   1   1
 'b'   0   1   1   1
 'c'   0   1   2   2
 'd'   0   1   2   2
 'e'   0   1   2   3
LCS = 3


### Ví dụ 3: khoảng cách sửa (edit distance) — số phép chèn/xóa/thay ít nhất

`dp[i][j]` = số phép sửa ít nhất biến `s[:i]` thành `t[:j]`.

- Ký tự cuối bằng nhau → `dp[i-1][j-1]`.
- Khác → `1 + min(thay: dp[i-1][j-1], xóa s[i-1]: dp[i-1][j], chèn t[j-1]: dp[i][j-1])`.
- Cơ sở: `dp[i][0] = i` (xóa hết), `dp[0][j] = j` (chèn hết).

In [6]:
def khoang_cach_sua(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

for a_, b_ in [("horse", "ros"), ("kitten", "sitting"), ("", "abc")]:
    print(f"{a_!r} → {b_!r}: {khoang_cach_sua(a_, b_)}")

'horse' → 'ros': 3
'kitten' → 'sitting': 3
'' → 'abc': 3


### Lỗi hay gặp

- `[[0] * n] * m` tạo `m` hàng là **cùng một** list → sửa một ô thì cả cột đổi.
- Lệch chỉ số: bảng `(n+1)×(m+1)` nhưng ký tự là `s[i-1]`, `t[j-1]`.
- Quên hàng/cột cơ sở (chuỗi rỗng, biên lưới).
- Khi tối ưu một hàng: ghi đè giá trị cần dùng lại (ví dụ `dp[i-1][j-1]`) — phải lưu tạm trước.

In [7]:
sai = [[0] * 3] * 2
sai[0][0] = 9
dung = [[0] * 3 for _ in range(2)]
dung[0][0] = 9
print("[[0]*3]*2 sau khi gán ô (0,0):", sai, "| tạo bằng vòng for:", dung)

[[0]*3]*2 sau khi gán ô (0,0): [[9, 0, 0], [9, 0, 0]] | tạo bằng vòng for: [[9, 0, 0], [0, 0, 0]]


## 2.3 Áp dụng vào bài này

Đây là ví dụ **edit distance** của DP hai chuỗi ở trên: `dp[i][j]` = số phép sửa ít nhất biến `word1[:i]` thành `word2[:j]`; ký tự cuối bằng nhau thì lấy `dp[i−1][j−1]`, khác thì `1 + min(thay, xóa, chèn)`.

# 3. Baseline: đệ quy

Nhìn vào ký tự **cuối** của hai chuỗi, `a = word1[:i]`, `b = word2[:j]`:

- Nếu `a[-1] == b[-1]`: không cần làm gì với ký tự cuối → `f(i-1, j-1)`.
- Ngược lại, ba lựa chọn cho ký tự cuối, cộng 1:
  - **Thay** `a[-1]` thành `b[-1]` → `f(i-1, j-1)`.
  - **Xóa** `a[-1]` → `f(i-1, j)`.
  - **Chèn** `b[-1]` vào cuối `a` → `f(i, j-1)`.

Trường hợp gốc: `f(0, j) = j` (chèn `j` ký tự), `f(i, 0) = i` (xóa `i` ký tự).

In [8]:
from typing import List
from functools import lru_cache

def edit_naive(word1: str, word2: str) -> int:
    calls = 0
    def f(i, j):
        nonlocal calls
        calls += 1
        if i == 0:
            return j
        if j == 0:
            return i
        if word1[i - 1] == word2[j - 1]:
            return f(i - 1, j - 1)
        return 1 + min(f(i - 1, j - 1), f(i - 1, j), f(i, j - 1))
    ans = f(len(word1), len(word2))
    return ans, calls

print(edit_naive("horse", "ros"))
print(edit_naive("abcdefgh", "hgfedcba"))

(3, 77)
(8, 123601)


Số lời gọi tăng **hàm mũ** vì cùng `(i, j)` bị tính đi tính lại. Chỉ có `(m+1)(n+1)` trạng thái khác nhau → **memo** hoặc **bảng**.

In [9]:
def edit_memo(word1: str, word2: str) -> int:
    @lru_cache(maxsize=None)
    def f(i, j):
        if i == 0:
            return j
        if j == 0:
            return i
        if word1[i - 1] == word2[j - 1]:
            return f(i - 1, j - 1)
        return 1 + min(f(i - 1, j - 1), f(i - 1, j), f(i, j - 1))
    return f(len(word1), len(word2))

print(edit_memo("horse", "ros"), edit_memo("intention", "execution"))

3 5


# 4. Ý tưởng chính: bảng DP

`dp[i][j]` = số thao tác ít nhất để biến `word1[:i]` thành `word2[:j]`.

```text
dp[0][j] = j,  dp[i][0] = i
nếu word1[i-1] == word2[j-1]:  dp[i][j] = dp[i-1][j-1]
ngược lại:                     dp[i][j] = 1 + min(dp[i-1][j-1],   # thay
                                                  dp[i-1][j],     # xóa
                                                  dp[i][j-1])     # chèn
```

Ba thao tác ↔ ba ô: **chéo trên trái** (thay), **trên** (xóa), **trái** (chèn).

In [10]:
def edit_table(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

print(edit_table("horse", "ros"), edit_table("intention", "execution"), edit_table("", "abc"))

3 5 3


# 5. Mô phỏng: bảng cho `"horse"` → `"ros"`

Hàng = tiền tố của `word1`, cột = tiền tố của `word2`.

In [11]:
def trace_edit_table(word1: str, word2: str):
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    print("      ε  " + "  ".join(word2))
    for i in range(m + 1):
        label = "ε" if i == 0 else word1[i - 1]
        print(f"   {label}  " + "  ".join(str(x) for x in dp[i]))
    return dp

_ = trace_edit_table("horse", "ros")

      ε  r  o  s
   ε  0  1  2  3
   h  1  1  2  3
   o  2  2  1  2
   r  3  2  2  2
   s  4  3  3  2
   e  5  4  4  3


Đọc vài ô:

| Ô | Ý nghĩa | Cách tính |
|---|---|---|
| `dp[1][1] = 1` | `"h"` → `"r"` | khác nhau → `1 + min(0, 1, 1)` (thay) |
| `dp[2][2] = 1` | `"ho"` → `"ro"` | `'o' == 'o'` → `dp[1][1] = 1` |
| `dp[4][3] = 2` | `"hors"` → `"ros"` | `'s' == 's'` → `dp[3][2] = 2` |
| `dp[5][3] = 3` | `"horse"` → `"ros"` | `'e' != 's'` → `1 + min(dp[4][2]=3, dp[4][3]=2, dp[5][2]=4)` = 3 (xóa `'e'`) |

# 6. Tối ưu bộ nhớ: hai hàng

Hàng `i` chỉ cần hàng `i-1` → giữ `prev` và `cur`. Bộ nhớ `O(n)`.

Có thể đổi vai để `n` là chuỗi **ngắn hơn** → `O(min(m, n))`.

In [12]:
def edit_two_rows(word1: str, word2: str) -> int:
    if len(word1) < len(word2):
        word1, word2 = word2, word1          # edit distance đối xứng
    n = len(word2)
    prev = list(range(n + 1))
    for i in range(1, len(word1) + 1):
        cur = [i] + [0] * n
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                cur[j] = prev[j - 1]
            else:
                cur[j] = 1 + min(prev[j - 1], prev[j], cur[j - 1])
        prev = cur
    return prev[n]

print(edit_two_rows("horse", "ros"), edit_two_rows("intention", "execution"))

3 5


Đổi chỗ được vì edit distance **đối xứng**: dãy thao tác biến `a → b` đảo ngược (chèn ↔ xóa) sẽ biến `b → a` với cùng số bước.

# 7. Phiên bản đúng format LeetCode

In [13]:
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        if len(word1) < len(word2):
            word1, word2 = word2, word1
        n = len(word2)
        prev = list(range(n + 1))
        for i in range(1, len(word1) + 1):
            cur = [i] + [0] * n
            for j in range(1, n + 1):
                if word1[i - 1] == word2[j - 1]:
                    cur[j] = prev[j - 1]
                else:
                    cur[j] = 1 + min(prev[j - 1], prev[j], cur[j - 1])
            prev = cur
        return prev[n]

In [14]:
import random

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.minDistance("horse", "ros"), 3)
check("Ví dụ 2", sol.minDistance("intention", "execution"), 5)
check("Hai chuỗi rỗng", sol.minDistance("", ""), 0)
check("Một chuỗi rỗng", sol.minDistance("abc", ""), 3)
check("Giống nhau", sol.minDistance("same", "same"), 0)
check("Khác hoàn toàn", sol.minDistance("abc", "xyz"), 3)
check("Chỉ cần chèn", sol.minDistance("ac", "abc"), 1)

random.seed(72)
for _ in range(3000):
    a = "".join(random.choice("abc") for _ in range(random.randint(0, 7)))
    b = "".join(random.choice("abc") for _ in range(random.randint(0, 7)))
    expected = edit_memo(a, b)
    assert sol.minDistance(a, b) == expected == edit_table(a, b) == sol.minDistance(b, a), (a, b)
print("3000 cặp ngẫu nhiên: khớp memo, bảng, và đối xứng")

Ví dụ 1                PASS  3
Ví dụ 2                PASS  5
Hai chuỗi rỗng         PASS  0
Một chuỗi rỗng         PASS  3
Giống nhau             PASS  0
Khác hoàn toàn         PASS  3
Chỉ cần chèn           PASS  1


3000 cặp ngẫu nhiên: khớp memo, bảng, và đối xứng

# 8. Vì sao đúng? Độ phức tạp?

Xét một dãy thao tác **tối ưu** biến `a = word1[:i]` thành `b = word2[:j]`. Nhìn vào số phận ký tự cuối `b[-1]` của đích:

- Nó **đến từ** `a[-1]` (giữ nguyên nếu bằng, hoặc bị thay) → phần còn lại là `a[:-1] → b[:-1]`.
- Nó **được chèn** → phần còn lại là `a → b[:-1]`.
- Nếu không thuộc hai trường hợp trên, `a[-1]` **bị xóa** → phần còn lại là `a[:-1] → b`.

Ba trường hợp phủ hết mọi khả năng, nên `min` của chúng là tối ưu. (Khi `a[-1] == b[-1]`, có thể chứng minh giữ nguyên luôn không tệ hơn.)

```text
Time:  O(m · n)
Space: O(m · n) với bảng | O(min(m, n)) với hai hàng
```

`500 × 500 = 250 000` ô — rất nhanh.

# 9. Những lỗi dễ mắc

## Lỗi 1: Khởi tạo hàng 0 / cột 0 bằng 0

`dp[i][0]` phải là `i` (xóa hết), không phải 0.

In [15]:
def wrong_init(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]      # SAI: quên dp[i][0] = i, dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

print("Sai :", wrong_init("horse", "ros"), "| Đúng:", sol.minDistance("horse", "ros"))
print("Sai :", wrong_init("abc", ""), "| Đúng:", sol.minDistance("abc", ""))

Sai :

 2 | Đúng: 3
Sai : 0 | Đúng: 3


## Lỗi 2: Lệch chỉ số giữa `dp` và chuỗi

`dp` có kích thước `(m+1) × (n+1)`; ô `dp[i][j]` ứng với ký tự `word1[i-1]`, `word2[j-1]`. Viết `word1[i] == word2[j]` sẽ so sai ký tự (và có thể `IndexError`).

## Lỗi 3: Cộng 1 cả khi hai ký tự bằng nhau

In [16]:
def wrong_always_plus(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            same = word1[i - 1] == word2[j - 1]
            dp[i][j] = min(dp[i - 1][j - 1] + 1, dp[i - 1][j] + 1, dp[i][j - 1] + 1)   # SAI: bỏ qua 'same'
    return dp[m][n]

print("Sai :", wrong_always_plus("same", "same"), "| Đúng: 0")

Sai : 4 | Đúng: 0


## Lỗi 4: Ở bản hai hàng, ghi đè `prev` khi còn cần

Nếu dùng **một** hàng duy nhất, `dp[i-1][j-1]` (ô chéo) đã bị ghi đè trước khi dùng. Cần lưu nó vào một biến tạm, hoặc dùng hai hàng như trên.

# 10. Mẹo nhận diện pattern

**DP trên hai chuỗi** khi đề so sánh / biến đổi hai chuỗi:

- `dp[i][j]` cho tiền tố `a[:i]`, `b[:j]`; bảng `(m+1) × (n+1)`.
- Nếu `a[i-1] == b[j-1]` → thường đi chéo.
- Ngược lại → kết hợp ba ô hàng xóm.

| Bài | Ô chéo khi bằng | Khi khác |
|---|---|---|
| #72 Edit Distance | `dp[i-1][j-1]` | `1 + min(3 ô)` |
| #1143 Longest Common Subsequence | `dp[i-1][j-1] + 1` | `max(trên, trái)` |
| #583 Delete Operation for Two Strings | `dp[i-1][j-1]` | `1 + min(trên, trái)` |
| #712 Minimum ASCII Delete Sum | `dp[i-1][j-1]` | `min(trên + ord, trái + ord)` |
| #115 Distinct Subsequences (Hard) | `dp[i-1][j-1] + dp[i-1][j]` | `dp[i-1][j]` |

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng DP cho `"cat"` → `"cut"` và `"sun"` → `"snow"`.

## Bài 2 — Điền chỗ trống (bảng DP)

In [17]:
def exercise_2_min_distance(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    # TODO: khởi tạo hàng 0, cột 0
    # TODO: điền bảng theo công thức
    return dp[m][n]


# assert exercise_2_min_distance("horse", "ros") == 3
# assert exercise_2_min_distance("intention", "execution") == 5
# assert exercise_2_min_distance("", "ab") == 2

## Bài 3 — In dãy thao tác

Trả về list các thao tác (dạng chuỗi) biến `word1` thành `word2`, độ dài đúng bằng edit distance. Định dạng: `"replace i a->b"`, `"delete i a"`, `"insert i b"`, với `i` là vị trí trong chuỗi **đang biến đổi** — để đơn giản, chỉ cần kiểm tra **số lượng** thao tác và việc áp dụng chúng cho ra `word2`. Hàm `apply_ops` có sẵn bên dưới để kiểm tra.

Gợi ý nhanh: truy vết ngược từ `dp[m][n]`; mỗi bước xác định đã đi từ ô nào. Áp dụng thao tác từ **cuối chuỗi về đầu** để chỉ số không bị lệch.

In [18]:
def apply_ops(word: str, ops: List[tuple]) -> str:
    s = list(word)
    for op in ops:
        kind, i = op[0], op[1]
        if kind == "replace":
            s[i] = op[2]
        elif kind == "delete":
            del s[i]
        else:  # insert
            s.insert(i, op[2])
    return "".join(s)


def exercise_3_edit_ops(word1: str, word2: str) -> List[tuple]:
    # TODO: trả list tuple ("replace", i, ch) | ("delete", i) | ("insert", i, ch)
    pass


# ops = exercise_3_edit_ops("horse", "ros"); assert len(ops) == 3 and apply_ops("horse", ops) == "ros"
# ops = exercise_3_edit_ops("intention", "execution"); assert len(ops) == 5 and apply_ops("intention", ops) == "execution"

## Bài 4 — Biến thể: Delete Operation for Two Strings (LeetCode #583)

Chỉ được **xóa** (trên cả hai chuỗi). Số bước ít nhất để hai chuỗi bằng nhau?

```python
min_delete("sea", "eat")   -> 2     # xóa 's' và 't'
min_delete("leetcode", "etco") -> 4
```

In [19]:
def exercise_4_min_delete(word1: str, word2: str) -> int:
    # TODO
    pass


# assert exercise_4_min_delete("sea", "eat") == 2
# assert exercise_4_min_delete("leetcode", "etco") == 4
# assert exercise_4_min_delete("", "abc") == 3

## Bài 5 — Mini challenge: One Edit Distance (LeetCode #161)

Kiểm tra `s` và `t` có cách nhau **đúng một** thao tác không, trong `O(n)` thời gian và `O(1)` bộ nhớ (không dùng bảng DP).

```python
is_one_edit("ab", "acb")  -> True
is_one_edit("", "")       -> False
is_one_edit("abc", "abd") -> True
is_one_edit("abc", "abc") -> False
```

In [20]:
def exercise_5_is_one_edit(s: str, t: str) -> bool:
    # TODO
    pass


# assert exercise_5_is_one_edit("ab", "acb") is True
# assert exercise_5_is_one_edit("", "") is False
# assert exercise_5_is_one_edit("abc", "abd") is True
# assert exercise_5_is_one_edit("abc", "abc") is False
# assert exercise_5_is_one_edit("a", "") is True

# 12. Gợi ý

**Bài 1**
- Mức 1: `"cat"` → `"cut"`: chỉ một ký tự khác → kết quả 1.

**Bài 3**
- Mức 1: tại `(i, j)`, nếu ký tự bằng nhau và `dp[i][j] == dp[i-1][j-1]` → đi chéo, không thêm thao tác.
- Mức 2: ngược lại, ô nào trong ba ô cho `dp[i][j] - 1` thì đó là thao tác đã dùng: chéo = replace tại `i-1`, trên = delete tại `i-1`, trái = insert `word2[j-1]` tại `i`.
- Mức 3: thu thập khi đi ngược → các thao tác đã theo thứ tự **từ cuối về đầu**, áp dụng theo thứ tự đó thì chỉ số vẫn đúng.

**Bài 4**
- Mức 1: như #72 nhưng bỏ ô chéo trong nhánh "khác".
- Mức 2: hoặc `len(a) + len(b) - 2 · LCS(a, b)`.

**Bài 5**
- Mức 1: nếu chênh độ dài > 1 → False. Cho `s` là chuỗi ngắn hơn.
- Mức 2: tìm vị trí khác đầu tiên `i`; nếu cùng độ dài so `s[i+1:] == t[i+1:]`, khác độ dài so `s[i:] == t[i+1:]`.
- Mức 3: nếu không có vị trí khác → True chỉ khi `len(t) == len(s) + 1`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
_ = trace_edit_table("cat", "cut")
print()
_ = trace_edit_table("sun", "snow")

      ε  c  u  t
   ε  0  1  2  3
   c  1  0  1  2
   a  2  1  1  2
   t  3  2  2  1

      ε  s  n  o  w
   ε  0  1  2  3  4
   s  1  0  1  2  3
   u  2  1  1  2  3
   n  3  2  1  2  3


## Đáp án Bài 2

In [22]:
def answer_2_min_distance(word1: str, word2: str) -> int:
    return edit_table(word1, word2)

assert answer_2_min_distance("horse", "ros") == 3
assert answer_2_min_distance("intention", "execution") == 5
assert answer_2_min_distance("", "ab") == 2
print("Bài 2 PASS (giống edit_table ở mục 4)")

Bài 2 PASS (giống edit_table ở mục 4)


## Đáp án Bài 3

In [23]:
def answer_3_edit_ops(word1: str, word2: str) -> List[tuple]:
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    ops = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and word1[i - 1] == word2[j - 1] and dp[i][j] == dp[i - 1][j - 1]:
            i, j = i - 1, j - 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            ops.append(("replace", i - 1, word2[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            ops.append(("delete", i - 1))
            i -= 1
        else:
            ops.append(("insert", i, word2[j - 1]))
            j -= 1
    return ops

ops = answer_3_edit_ops("horse", "ros")
print(ops)
assert len(ops) == 3 and apply_ops("horse", ops) == "ros"
ops = answer_3_edit_ops("intention", "execution"); assert len(ops) == 5 and apply_ops("intention", ops) == "execution"
for _ in range(2000):
    a = "".join(random.choice("abc") for _ in range(random.randint(0, 7)))
    b = "".join(random.choice("abc") for _ in range(random.randint(0, 7)))
    ops = answer_3_edit_ops(a, b)
    assert len(ops) == sol.minDistance(a, b) and apply_ops(a, ops) == b, (a, b)
print("Bài 3 PASS (2000 cặp: đúng số thao tác và áp dụng ra đúng word2)")

[('delete', 4), ('delete', 2), ('replace', 0, 'r')]


Bài 3 PASS (2000 cặp: đúng số thao tác và áp dụng ra đúng word2)


Vì sao áp dụng theo thứ tự truy vết lại đúng chỉ số? Khi đi ngược, các thao tác được ghi theo vị trí `i` **giảm dần** trong `word1`. Thay đổi ở vị trí lớn không làm lệch các vị trí nhỏ hơn được xử lý sau.

## Đáp án Bài 4

In [24]:
def answer_4_min_delete(word1: str, word2: str) -> int:
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

assert answer_4_min_delete("sea", "eat") == 2
assert answer_4_min_delete("leetcode", "etco") == 4
assert answer_4_min_delete("", "abc") == 3
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [25]:
def answer_5_is_one_edit(s: str, t: str) -> bool:
    if len(s) > len(t):
        s, t = t, s
    if len(t) - len(s) > 1:
        return False
    for i in range(len(s)):
        if s[i] != t[i]:
            if len(s) == len(t):
                return s[i + 1:] == t[i + 1:]
            return s[i:] == t[i + 1:]
    return len(t) == len(s) + 1

assert answer_5_is_one_edit("ab", "acb") is True
assert answer_5_is_one_edit("", "") is False
assert answer_5_is_one_edit("abc", "abd") is True
assert answer_5_is_one_edit("abc", "abc") is False
assert answer_5_is_one_edit("a", "") is True
for _ in range(3000):
    a = "".join(random.choice("ab") for _ in range(random.randint(0, 5)))
    b = "".join(random.choice("ab") for _ in range(random.randint(0, 5)))
    assert answer_5_is_one_edit(a, b) == (sol.minDistance(a, b) == 1), (a, b)
print("Bài 5 PASS (khớp minDistance == 1 trên 3000 cặp)")

Bài 5 PASS (khớp minDistance == 1 trên 3000 cặp)


(Phép so sánh lát cắt `s[i+1:] == t[i+1:]` tạo bản sao — về lý thuyết `O(n)` bộ nhớ phụ; để thật sự `O(1)` có thể so bằng vòng lặp chỉ số.)

# 14. Tổng kết

> `dp[i][j]` = edit distance của `word1[:i]` và `word2[:j]`. Bằng nhau → đi chéo; khác → `1 + min(chéo=thay, trên=xóa, trái=chèn)`. Hàng/cột 0 là `0, 1, 2, …`.

---

## Checklist trước khi nộp

- [ ] Bảng `(m+1) × (n+1)`, ký tự ứng với `i-1`, `j-1`?
- [ ] `dp[i][0] = i`, `dp[0][j] = j`?
- [ ] Không cộng 1 khi hai ký tự bằng nhau?
- [ ] Bản tiết kiệm bộ nhớ không ghi đè ô chéo trước khi dùng?